# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

This section checks two signals before defining the baseline rule. The signals are chosen from the fixed Week 3 feature contract and are intended to support the Refresh / Content Opportunity Scoring lane.

In [1]:
# Cell 2 — Setup

import os
import sys
import subprocess
import getpass
from pathlib import Path

# Install required packages if needed
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "duckdb", "pandas", "numpy"],
    check=True
)

import duckdb
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("Setup complete.")

Python: 3.13.15
Setup complete.


In [2]:
# Cell 3 — Connect to the fixed FlyRank warehouse release

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content/**/*.parquet')",

    "dim_clients":
        f"read_parquet('{REL}/dim_clients/**/*.parquet')"
}

print("Connected to FlyRank warehouse.")
print("Using fixed Week 3 data contract.")
print("Development month: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.
Using fixed Week 3 data contract.
Development month: March 2026


In [3]:
# Cell 4 — Fixed Week 3 feature contract

FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update"
]

CONTEXT = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Fixed features:")
for feature in FEATURES:
    print(" -", feature)

print("\nContext fields:")
for field in CONTEXT:
    print(" -", field)

Fixed features:
 - imp_prev30
 - clicks_prev30
 - avg_position_prev30
 - content_age_days
 - days_since_last_update

Context fields:
 - client_hash_id
 - content_hash_id
 - report_date


### Signal 1 — Staleness

I will use `days_since_last_update` as the first signal. The reasoning is that older content may deserve a refresh review because facts, dates, links, or other information can become outdated. This signal is directly connected to the Content Refresh intervention discussed in Week 4.

I will use 180 days as the initial operational threshold. This is a transparent starting threshold for the baseline rather than a fitted model parameter.

### Signal 2 — Recent search visibility

I will use `imp_prev30` as the second signal. A page with meaningful recent impressions has observable search visibility, so reviewing a stale page with substantial visibility may have more practical value than spending the same review effort on a page with almost no visibility.

I will use 500 previous-30-day impressions as the initial visibility threshold. This is a simple operational threshold for the baseline and will be checked against the observed bucket counts before being treated as useful.

The two signals are intentionally simple and interpretable. The goal of this baseline is not to optimize weights but to create a transparent rule that a later ML model must beat.


In [4]:
# Cell 6 — Build the fixed five-feature decision table

decision_date = "2026-03-31"

feature_query = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-02'
      AND report_date <= DATE '2026-03-31'
),

performance AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev30,
        SUM(gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_avg_position * gsc_impressions)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM daily
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM performance
"""

features_df = con.sql(feature_query).df()

print("Feature table shape:", features_df.shape)
print("\nColumns:")
print(features_df.columns.tolist())

features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table shape: (331436, 5)

Columns:
['client_hash_id', 'content_hash_id', 'imp_prev30', 'clicks_prev30', 'avg_position_prev30']


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30
0,client_62f4a7e64f5e0096,content_5d562ebeda84bdc7,773.0,0.0,11.467012
1,client_62f4a7e64f5e0096,content_d6f555d072e070be,25534.0,72.0,4.089332
2,client_62f4a7e64f5e0096,content_dabbfdf80d8773f0,1448.0,9.0,4.669199
3,client_62f4a7e64f5e0096,content_a761d62e362213d3,5595.0,24.0,4.188382
4,client_62f4a7e64f5e0096,content_eb6738715ef9bac1,412.0,2.0,9.446602


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.